In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5

In [3]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***loading conv info***

In [4]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

***loading llama labled messages***

In [5]:
messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

***loading llama labled conversation chunks***

In [ ]:
messages_with_lables_path = 'trainDatasets/message_chunks_synthetic_results.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

***loading all messages***

In [6]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


## creating Dataset objects for train and test

***defining makers***

In [7]:
# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_weighted_train_loaders(dfs, weights, tokenize):
    #make sampled df
    acc_train = pd.DataFrame(data= {})
    for df, weight in zip(dfs,weights):
        sample = df.sample(frac = weight)
        acc_train = pd.concat([acc_train,sample[["anonymized" , "label"]]])
    
    #create datasets
    train_dataset = Dataset.from_pandas(acc_train)
    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader

def make_test_loader(df, tokenize):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return test_loader

***making***

In [8]:
#labled_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [1,0], tokenize)
#all_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [0,1], tokenize)
labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)
all_messages_test_loader = make_test_loader(test_all_messages_df, tokenize)

Map: 100%|██████████| 6047/6047 [00:04<00:00, 1263.26 examples/s]


***for labeled messages***

In [8]:
'''
# creating Dataset objects
train_labled_messages_dataset = Dataset.from_pandas(train_labled_messages_df)
test_labled_messages_dataset = Dataset.from_pandas(test_labled_messages_df)

train_labled_messages_dataset = train_labled_messages_dataset.map(tokenize, batched=True, batch_size=16)
test_labled_messages_dataset = test_labled_messages_dataset.map(tokenize, batched=True, batch_size=16)

# setting the format to pytorch tensors
train_labled_messages_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_labled_messages_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

labled_messages_train_loader = DataLoader(train_labled_messages_dataset, batch_size=batch_size, shuffle=True)
labled_messages_test_loader = DataLoader(test_labled_messages_dataset, batch_size=batch_size, shuffle=False)
'''

Map: 100%|██████████| 83832/83832 [00:25<00:00, 3230.73 examples/s]


***for concat messages***

In [9]:
'''# creating Dataset objects
train_all_messages_dataset = Dataset.from_pandas(train_all_messages_df)
test_all_messages_dataset = Dataset.from_pandas(test_all_messages_df)

train_all_messages_dataset = train_all_messages_dataset.map(tokenize, batched=True, batch_size=16)
test_all_messages_dataset = test_all_messages_dataset.map(tokenize, batched=True, batch_size=16)

# setting the format to pytorch tensors
train_all_messages_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_all_messages_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

all_messages_train_loader = DataLoader(train_all_messages_dataset, batch_size=batch_size, shuffle=True)
all_messages_test_loader = DataLoader(test_all_messages_dataset, batch_size=batch_size, shuffle=False)
'''

Map: 100%|██████████| 6047/6047 [00:04<00:00, 1253.85 examples/s]


## train model

***train on labled messages***

In [9]:
def general_trainer(train_loader, loss_fn=None):
    optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
    bert_model.train()
    
    total_batches = len(train_loader)
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        if loss_fn is None:
            loss = outputs.loss
        else:
            loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")

In [10]:
plan = [
    [0.9,0.1],
    [0.8,0.2],
    [0.65,0.3],
    [0.1,0.7],
    [0.05,0.9],
    [0.025,0.95]
]
for w in plan:
    train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , w, tokenize)
    general_trainer(train_loader)

Map: 100%|██████████| 295327/295327 [01:26<00:00, 3419.82 examples/s]


Batch 1000/18458 completed. 17458 remaining.
Batch 2000/18458 completed. 16458 remaining.
Batch 3000/18458 completed. 15458 remaining.
Batch 4000/18458 completed. 14458 remaining.
Batch 5000/18458 completed. 13458 remaining.
Batch 6000/18458 completed. 12458 remaining.
Batch 7000/18458 completed. 11458 remaining.
Batch 8000/18458 completed. 10458 remaining.
Batch 9000/18458 completed. 9458 remaining.
Batch 10000/18458 completed. 8458 remaining.
Batch 11000/18458 completed. 7458 remaining.
Batch 12000/18458 completed. 6458 remaining.
Batch 13000/18458 completed. 5458 remaining.
Batch 14000/18458 completed. 4458 remaining.
Batch 15000/18458 completed. 3458 remaining.
Batch 16000/18458 completed. 2458 remaining.
Batch 17000/18458 completed. 1458 remaining.
Batch 18000/18458 completed. 458 remaining.


Map: 100%|██████████| 265200/265200 [01:20<00:00, 3277.82 examples/s]


Batch 1000/16575 completed. 15575 remaining.
Batch 2000/16575 completed. 14575 remaining.
Batch 3000/16575 completed. 13575 remaining.
Batch 4000/16575 completed. 12575 remaining.
Batch 5000/16575 completed. 11575 remaining.
Batch 6000/16575 completed. 10575 remaining.
Batch 7000/16575 completed. 9575 remaining.
Batch 8000/16575 completed. 8575 remaining.
Batch 9000/16575 completed. 7575 remaining.
Batch 10000/16575 completed. 6575 remaining.
Batch 11000/16575 completed. 5575 remaining.
Batch 12000/16575 completed. 4575 remaining.
Batch 13000/16575 completed. 3575 remaining.
Batch 14000/16575 completed. 2575 remaining.
Batch 15000/16575 completed. 1575 remaining.
Batch 16000/16575 completed. 575 remaining.


Map: 100%|██████████| 218801/218801 [01:05<00:00, 3315.56 examples/s]


...

Batch 1000/3093 completed. 2093 remaining.
Batch 2000/3093 completed. 1093 remaining.
Batch 3000/3093 completed. 93 remaining.


Map: 100%|██████████| 38039/38039 [00:21<00:00, 1769.32 examples/s]


Batch 1000/2378 completed. 1378 remaining.
Batch 2000/2378 completed. 378 remaining.


Map: 100%|██████████| 31112/31112 [00:20<00:00, 1544.19 examples/s]


Batch 1000/1945 completed. 945 remaining.


In [10]:
'''
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
bert_model.train()

#progress_bar = tqdm(range(epochs * len(labled_messages_train_loader)), desc="Training")
total_batches = len(labled_messages_train_loader)
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for i, batch in enumerate(labled_messages_train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"  [Epoch {epoch + 1}] Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")
'''


Epoch 1/3
  [Epoch 1] Batch 1000/20341 completed. 19341 remaining.
  [Epoch 1] Batch 2000/20341 completed. 18341 remaining.
  [Epoch 1] Batch 3000/20341 completed. 17341 remaining.
  [Epoch 1] Batch 4000/20341 completed. 16341 remaining.
  [Epoch 1] Batch 5000/20341 completed. 15341 remaining.
  [Epoch 1] Batch 6000/20341 completed. 14341 remaining.
  [Epoch 1] Batch 7000/20341 completed. 13341 remaining.
  [Epoch 1] Batch 8000/20341 completed. 12341 remaining.
  [Epoch 1] Batch 9000/20341 completed. 11341 remaining.
  [Epoch 1] Batch 10000/20341 completed. 10341 remaining.
  [Epoch 1] Batch 11000/20341 completed. 9341 remaining.
  [Epoch 1] Batch 12000/20341 completed. 8341 remaining.
  [Epoch 1] Batch 13000/20341 completed. 7341 remaining.
  [Epoch 1] Batch 14000/20341 completed. 6341 remaining.
  [Epoch 1] Batch 15000/20341 completed. 5341 remaining.
  [Epoch 1] Batch 16000/20341 completed. 4341 remaining.
  [Epoch 1] Batch 17000/20341 completed. 3341 remaining.
  [Epoch 1] Batch 1

***train on concat messages***

In [34]:
'''
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
bert_model.train()

#progress_bar = tqdm(range(epochs * len(labled_messages_train_loader)), desc="Training")
total_batches = len(all_messages_train_loader)
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for i, batch in enumerate(all_messages_train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"  [Epoch {epoch + 1}] Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")
'''


Epoch 1/3
  [Epoch 1] Batch 1000/1512 completed. 512 remaining.

Epoch 2/3
  [Epoch 2] Batch 1000/1512 completed. 512 remaining.

Epoch 3/3
  [Epoch 3] Batch 1000/1512 completed. 512 remaining.


## eval model

In [11]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

***eval messages***

In [13]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.9516294493749403
Precision: 0.8226206896551724
Recall: 0.6829268292682927
F1: 0.74629293624476
ROC-AUC: 0.9695915190185529
F2: 0.706936608031862


***eval conversation***

In [12]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in all_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.8814288076732264
Precision: 0.6381578947368421
Recall: 0.7972602739726027
F1: 0.7088915956151035
ROC-AUC: 0.9206584120801706
F2: 0.7593945720250522


In [14]:
name_for_save = input("enter model name: ")
torch.save(bert_model.state_dict(), f"model_weights_{name_for_save}.pth")

enter model name:  progressive_curiculum_learning_conv_messages
